# 242. Valid Anagram
**Difficulty:** 🟢 Easy · **Topic:** String · **LeetCode:** https://leetcode.com/problems/valid-anagram/

## 💡 Concepts

**Core concept(s):** Counting letters (a **hash map**), or **sorting** both strings.

**Why it applies here:** Two words are anagrams when they use the exact same letters the same number of times. So either count each letter and compare the counts, or sort both words — anagrams become identical once sorted.

**Key intuition:** Don't compare the words directly — compare their *letter recipes*.

---

### 📚 What is a Hash Map / Hash Set?
A **hash map** (Python `dict`) stores **key → value** pairs; a **hash set** (`set`) stores unique keys. Both use a *hash function* to jump straight to a slot instead of scanning.
- **Operations & complexity:** insert / lookup / delete are **O(1) on average**.
- **In Python:** `dict` for counts/mappings, `set` for "have I seen this?". `collections.Counter` counts items for you.

### 📚 What is Sorting (as a tool)?
**Sorting** reorders characters so structure becomes obvious (e.g. two anagrams become the *same* string once sorted).
- **Complexity:** Python's `sorted()` is **O(k log k)** for length k.

---

**Prerequisite knowledge:**
- Counting with a `dict`/`Counter`.
- Sorting strings.

## 📝 Problem

Return `True` if `t` is an anagram of `s` (same letters, rearranged).

**Example**
```
s = "anagram", t = "nagaram"  -> True
s = "rat",     t = "car"      -> False
```

> Two meaningfully distinct approaches: sorting `O(n log n)` and letter-counting `O(n)`.

### Approach 1 — Sort Both (worst)

**Idea:** Sort both strings; anagrams become the same string.

**Time complexity:** `O(n log n)` — the sort dominates.

**Space complexity:** `O(n)` for the sorted copies.

In [ ]:
def is_anagram_sort(s: str, t: str) -> bool:
    # Two words are anagrams if they contain the exact same letters.
    # Sorting each word lines its letters up in the same order, so
    # anagrams become identical strings we can compare directly.
    return sorted(s) == sorted(t)          # e.g. sorted("nag") == sorted("gan")

### Approach 2 — Count Letters (optimal)

**Idea:** Count how many of each letter each word has; they're anagrams iff the counts match. Different lengths can't match, so check that first.

**Time complexity:** `O(n)`.

**Space complexity:** `O(1)` — at most 26 lowercase letters (a fixed-size count).

In [ ]:
def is_anagram_count(s: str, t: str) -> bool:
    # Quick reject: different lengths can never be anagrams.
    if len(s) != len(t):
        return False
    count = {}                             # letter -> how many times it appears in s
    for c in s:                            # step 1: tally every letter of the first word
        count[c] = count.get(c, 0) + 1     # add one to this letter's running total
    for c in t:                            # step 2: "spend" one of each letter for word t
        if c not in count:                 # t needs a letter that s never had -> not anagram
            return False
        count[c] -= 1                      # use up one occurrence of this letter
        if count[c] == 0:                  # exhausted this letter -> drop it from the map
            del count[c]
    return len(count) == 0                 # anagram only if every letter was used up exactly

In [ ]:
# Correctness check
tests = [("anagram","nagaram",True), ("rat","car",False), ("a","ab",False), ("","",True)]
for s, t, exp in tests:
    a, b = is_anagram_sort(s, t), is_anagram_count(s, t)
    print(f"{s!r} vs {t!r} -> sort={a}, count={b} | expected={exp}")
    assert a == b == exp, "mismatch!"
print("\nAll tests passed")

## ⏱️ Empirically Checking the Complexities

Big-O can't be read off a function directly, but it can be **measured**. We time each approach on inputs of growing `n` and read the **doubling ratio** — how much runtime grows when `n` doubles.

| Theoretical | Ratio when `n` → `2n` |
|-------------|-----------------------|
| `O(n)`        | ≈ **2×** |
| `O(n log n)`  | ≈ **2×** (slightly more) |
| `O(n²)`       | ≈ **4×** |
| `O(n³)`       | ≈ **8×** |

Inputs are built to force the **worst case** (no early exit). Sub-millisecond rows are noisy — look at the trend.

In [ ]:
import os, sys
_root = os.getcwd()
for _ in range(5):
    if os.path.exists(os.path.join(_root, "bench_utils.py")):
        break
    _root = os.path.dirname(_root)
if _root not in sys.path:
    sys.path.insert(0, _root)
from bench_utils import benchmark   # shared: prints ratio table + optional log-log plot

def make_worst_case(n):
    s = ("abcde" * (n // 5 + 1))[:n]
    return (s, s)                           # equal anagrams -> full work, no early exit

solutions = {
    "sort  O(n log n)": is_anagram_sort,
    "count O(n)      ": is_anagram_count,
}
sizes = [2000, 4000, 8000, 16000]

benchmark(solutions, make_worst_case, sizes, plot=True)


## 🧩 Patterns Learned

- **Compare recipes, not items:** counting letters (a frequency map) reduces "are these the same multiset?" to comparing counts — `O(n)`.
- **Sort to normalize:** sorting maps every anagram to one canonical form; slower but tiny to write.
- **Signal:** "same letters rearranged", "permutation of", "same characters".
- **Related problems:** Group Anagrams, Valid Parentheses (matching), Find All Anagrams in a String.
- **Common pitfalls:** (1) forgetting the length check; (2) assuming only lowercase when input may have unicode/spaces.